# Lab 8 – NumPy, pandas, and Data Exploration
**Dataset:** Movie posters and metadata from TMDb API (stored in MongoDB `image_metadata` collection)


In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.analytics.data_loader   import load_from_mongo, save_to_csv, load_from_csv, compute_mean_across_chunks, process_chunks_per_language, optimise_dtypes, memory_report
from src.analytics.numpy_ops     import create_arrays_demo, vectorized_ops
from src.analytics.explorer      import inspect_shape, display_info, describe_stats, value_count_report, extract_release_year, save_distribution_charts
from src.analytics.selector      import select_columns, sample_rows, filter_quality_and_popularity, filter_by_language, filter_by_range
from src.analytics.regex_ops     import extract_year_from_title, filter_titles_by_prefix, count_crime_terms, find_short_overviews, parse_genres, top_genres, validate_movie_ids
from src.analytics.quality_report import missing_value_report, detect_zero_as_missing, detect_outliers_iqr, full_quality_audit, save_quality_report

print('All imports OK')

## Part 2 – NumPy Foundations
Array creation using 6 methods + vectorized operations (no Python loops).

In [ ]:
arrays = create_arrays_demo()
for name, arr in arrays.items():
    print(f'{name:20s}  shape={arr.shape}  dtype={arr.dtype}  ndim={arr.ndim}  size={arr.size}')

## Part 3 – Loading & Memory Management

In [ ]:
# Load from MongoDB
df = load_from_mongo('image_metadata')
print(f'Loaded {len(df)} rows, {len(df.columns)} columns')
df.head(3)

In [ ]:
# Export to CSV
csv_path = save_to_csv(df)
print(f'Saved to: {csv_path}')

In [ ]:
# Global mean across chunks
global_mean = compute_mean_across_chunks(csv_path, 'vote_average', chunksize=3)
print(f'Global mean vote_average (across chunks): {global_mean:.3f}')

In [ ]:
# Per-language stats across chunks
lang_summary = process_chunks_per_language(csv_path, chunksize=3)
lang_summary

In [ ]:
# dtype optimisation + memory report
df_opt = optimise_dtypes(df)
mem    = memory_report(df, df_opt)
print(f"Before: {mem['before_mb']:.4f} MB  |  After: {mem['after_mb']:.4f} MB  |  Reduction: {mem['reduction_pct']}%")

## Part 2 (cont.) – Vectorized NumPy on real data

In [ ]:
stats, normalized = vectorized_ops(df)
print('vote_average stats:', stats['vote_average'])
print('popularity stats:',   stats['popularity'])
print('Normalized ratings (first 5):', normalized[:5])

## Part 4 – Exploratory Data Analysis

In [ ]:
shape_info = inspect_shape(df)
print(shape_info)

In [ ]:
print(display_info(df))

In [ ]:
describe_stats(df)

In [ ]:
cat_report = value_count_report(df, ['original_language', 'source', 'type'])
for col, info in cat_report.items():
    print(f"\n{col}  (nunique={info['nunique']})")
    print(info['value_counts'])

In [ ]:
df_year = extract_release_year(df)
df_year[['title', 'release_date', 'release_year']].head(10)

In [ ]:
chart_paths = save_distribution_charts(df)
print('Charts saved:')
for p in chart_paths:
    print(' ', p)

# Display in notebook
from IPython.display import Image, display
for p in chart_paths:
    display(Image(str(p)))

## Part 5 – loc, iloc, Boolean Filtering

In [ ]:
# Column selection
cols = select_columns(df, ['title', 'vote_average', 'popularity', 'original_language', 'release_date'])
cols.head()

In [ ]:
# iloc sample
sample = sample_rows(df, 0, 5)
print(f'iloc[0:5] => {len(sample)} rows')
sample[['title','vote_average']]

In [ ]:
# loc boolean: high quality
hq = filter_quality_and_popularity(df, min_rating=6.5, min_popularity=5.0)
print(f'High quality (rating>=6.5, pop>=5): {len(hq)} rows')
hq[['title','vote_average','popularity']]

In [ ]:
# isin filter
eng = filter_by_language(df, ['en'])
non_eng = filter_by_language(df, ['en'], exclude=True)
print(f'English: {len(eng)}  |  Non-English: {len(non_eng)}')

In [ ]:
# between filter
mid = filter_by_range(df, 'vote_average', 6.0, 8.0)
print(f'vote_average between 6-8: {len(mid)} rows')
mid[['title','vote_average']]

## Part 6 – Regular Expressions

In [ ]:
if 'title' in df.columns:
    years  = extract_year_from_title(df['title'])
    the_t  = filter_titles_by_prefix(df['title'], 'The')
    print('Years in titles:\n', years.dropna())
    print('\nTitles starting with "The":\n', the_t.values)

In [ ]:
if 'overview' in df.columns:
    crime = count_crime_terms(df['overview'])
    short = find_short_overviews(df['overview'])
    print('Crime term counts per movie:\n', crime.values)
    print(f'\nShort overviews (<60 chars): {len(short)}')
    print(short.values)

In [ ]:
genres  = parse_genres(df)
top_g   = top_genres(df, 10)
print('Genre strings (first 5):\n', genres.head().values)
print('\nTop genres:', top_g)

In [ ]:
if 'movie_id' in df.columns:
    valid_ids = validate_movie_ids(df['movie_id'])
    print(f'Valid TMDB IDs: {valid_ids.sum()}/{len(valid_ids)}')

## Part 7 – Data Quality Assessment

In [ ]:
missing = missing_value_report(df)
missing

In [ ]:
zeros = detect_zero_as_missing(df)
zeros

In [ ]:
outliers = detect_outliers_iqr(df)
outliers

In [ ]:
# Full audit + save
audit = full_quality_audit(df)
report_path = save_quality_report(audit)
print(f'Quality report saved to: {report_path}')
audit

In [ ]:
# Show missing heatmap
from IPython.display import Image
Image('data/processed/analytics/charts/missing_data_heatmap.png')